# NB2e — AI Counter-Generation: GPT-5 Mini (via OpenRouter)

The fifth generator, and the first of the two that close out the corpus. Same synchronous structure
as my DeepSeek and Qwen notebooks — **not** the Batch design I used for Sonnet/Opus, because
OpenRouter has no batch endpoint. That turns out to be an advantage: synchronous calls give me the
regenerate loop back, and that loop is worth a lot (DeepSeek and Qwen both landed 97.6% per-card
pass rates with it, versus 92.8% for Sonnet and 80.9% for Opus through Batch, which can't retry).

**Input:** `fact_cards.parquet` + all five prior outputs.
**Output:** `aig_gpt.parquet` — 440 passing articles.

## Why GPT-5 Mini and not GPT-5 or GPT-5.5

The most capable model is not the best model for this task, and I have direct evidence: Opus 4.8 is
stronger than Sonnet 5 on every public benchmark, yet on my gate it failed **90/470** against
Sonnet's **34/470**. The reason showed up cleanly in the reject analysis — entity coverage was fine
for both (93% vs 89%), but **number coverage collapsed for Opus (30% vs 45%)**. The stronger model
writes more editorially: it renders "37 قتيلاً" as "عشرات القتلى" because that reads better. For
counter-generation I need the opposite — literal carry-over of the card's facts.

So I deliberately pick the efficiency tier. OpenAI's own description of GPT-5 Mini is exactly the
property I want: the same instruction-following as GPT-5, at lower latency and ~1/5 the cost.

## Draw order: carry-over first, then fresh

I draw from one ordered list and stop when I've banked **440 successes**. Rejects don't count toward
the quota — they flow to Gemini, which cleans up whatever is left. Carry-over rejects from earlier
generators go **first** in the list so they get the earliest, freshest attempts.

## Setup, secrets, config

In [1]:
!pip -q install openai >/dev/null 2>&1

import pandas as pd, numpy as np, json, re, os, time, random, glob
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# OpenRouter is OpenAI-API-compatible. The secret MUST be attached to THIS notebook
# (Add-ons -> Secrets). A fresh notebook does not inherit secrets automatically.
API_KEY = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
client   = OpenAI(api_key=API_KEY, base_url='https://openrouter.ai/api/v1',
                  timeout=120.0, max_retries=0)   # 120s hard cap; I do my own retries

MODEL_NAME   = 'openai/gpt-5-mini'   # efficiency tier on purpose — see the header note
GENERATOR_ID = 'gpt'
QUOTA        = 440                   # successes to bank; rejects flow to Gemini
CARDS_PATH   = '/kaggle/input/notebooks/bahaaqassem/nb2c-build-fact-cards/fact_cards.parquet'
OUT_DIR      = '/kaggle/working'

# Every prior generator's output. I read these to (a) skip cards already processed and
# (b) pick up rejects nobody has salvaged yet. IMPORTANT: aigt-aig-qwen must hold the FINAL
# 637-row Qwen file — an earlier run read a stale 529-row copy and duplicated 2 cards.
PRIOR_OUTPUTS = [
    '/kaggle/input/notebooks/bahaaqassem/nb2d-generate-deepseek/aig_deepseek.parquet',
    '/kaggle/input/datasets/bahaaqassem/aig-qwen/aig_qwen.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2f-generate-sonnet/aig_sonnet.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2g-generate-opus/aig_opus.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2h-retry-rejects/aig_sonnet_retry.parquet',
]

RESUME_PATH  = None          # set to a partial aig_gpt.parquet to resume after a timeout
PILOT_N      = 30
PILOT_ONLY   = False          # True = 30-article pilot only. Set False for the production run.

PRICE_IN, PRICE_OUT = 0.25, 2.00     # GPT-5 Mini $/1M tokens (OpenRouter), for cost projection

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})')
            return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True):
        print('   ', p)
    raise FileNotFoundError(f'could not find {preferred} (looked for {keywords})')

CARDS_PATH = find_parquet(CARDS_PATH, 'fact_cards', 'fact-cards')
cards = pd.read_parquet(CARDS_PATH)
print('cards:', cards.shape)

cards: (3500, 16)


## Work list — carry-over first, then a sequential draw

One ordered list (seeded shuffle, identical in every notebook). I skip every card any prior
generator already processed, and I put the **unsalvaged rejects at the front** so they get tried
early rather than after 400 fresh cards. Then I generate down the list until I've banked QUOTA
successes; rejects don't count, so they simply flow on to Gemini.

`failed - succeeded` is what makes the relay self-correcting across notebooks — DeepSeek's original
22 rejects don't appear here at all, because Qwen rescued every one of them.

In [2]:
ordered = cards.sample(frac=1.0, random_state=42).reset_index(drop=True)

processed, succeeded, failed = set(), set(), set()
for path in PRIOR_OUTPUTS:
    if not os.path.exists(path):
        kw = os.path.basename(path).replace('.parquet', '')
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True)
                if kw in p or kw.replace('_', '-') in p]
        path = hits[0] if hits else path
    if os.path.exists(path):
        prev = pd.read_parquet(path)
        processed |= set(prev['source_pair_id'])
        succeeded |= set(prev.loc[prev['gate_passed'], 'source_pair_id'])
        failed    |= set(prev.loc[~prev['gate_passed'], 'source_pair_id'])
        print(f'{os.path.basename(path):<28} {len(prev):4d} rows, '
              f'{int((~prev["gate_passed"]).sum()):3d} rejects')
    else:
        print(f'WARNING: prior output not found: {path}')

carry_over  = failed - succeeded                       # failed somewhere, rescued nowhere
carry_cards = ordered[ordered['pair_id'].isin(carry_over)]
fresh_cards = ordered[~ordered['pair_id'].isin(processed)]

my_cards = (pd.concat([carry_cards, fresh_cards])       # carry-over FIRST
              .drop_duplicates('pair_id')
              .reset_index(drop=True))

print(f'\ncarry-over (unsalvaged): {len(carry_cards)}')
print(f'fresh (untouched)      : {len(fresh_cards)}')
print(f'work list              : {len(my_cards)} cards | quota {QUOTA} successes')
print('sample ids:', my_cards['pair_id'].head(3).tolist())

aig_deepseek.parquet          900 rows,  22 rejects
aig_qwen.parquet              529 rows,  11 rejects
aig_sonnet.parquet            470 rows,  34 rejects
aig_opus.parquet              470 rows,  90 rejects
aig_sonnet_retry.parquet      485 rows,  81 rejects

carry-over (unsalvaged): 81
fresh (untouched)      : 803
work list              : 884 cards | quota 440 successes
sample ids: ['HA_02878', 'HA_00246', 'HA_02794']


## Prompt (mirror rule + explicit length band + no-formatting)

Conditional requirements fire only when the source actually has that element (mirror rule). I state
an explicit length band and forbid markdown so the raw output already matches the human corpus
(single paragraph, no `**`).

In [3]:
STYLE_HINTS = [
    'ابدأ بفقرة استهلالية تلخّص الحدث',
    'اذكر خلفية موجزة للسياق',
    'أورد ردود فعل الأطراف المعنية',
    'اختم بما هو متوقّع أو منتظر',
    'استخدم بنية الهرم المقلوب',
    'انسب المعلومات إلى مصادرها',
]

SYSTEM_PROMPT = (
    'أنت صحفي محترف يكتب تقارير إخبارية بالعربية الفصحى لصالح غرفة أخبار محترمة. '
    'تكتب بأسلوب صحفي رصين ودقيق، وتلتزم بالحقائق المعطاة دون إضافة معلومات من خارجها.'
)

# ---- number cleanup so prompt & gate agree (dual-calendar months, junk single digits) ----
MONTH_GROUPS = [
    {'يناير','كانون الثاني'}, {'فبراير','شباط'}, {'مارس','آذار'},
    {'أبريل','نيسان'}, {'مايو','أيار'}, {'يونيو','حزيران'},
    {'يوليو','تموز'}, {'أغسطس','آب'}, {'سبتمبر','أيلول'},
    {'أكتوبر','تشرين الأول'}, {'نوفمبر','تشرين الثاني'}, {'ديسمبر','كانون الأول'},
]
AMBIG = {'كانون': 0, 'تشرين': 9}

def month_group(item):
    for i, g in enumerate(MONTH_GROUPS):
        if any(m in item for m in g):
            return i
    for k, i in AMBIG.items():
        if k in item:
            return i
    return None

def clean_numbers(nums, cap=6):
    seen, out = set(), []
    for n in nums:
        n = n.strip()
        if re.fullmatch(r'[٠-٩0-9]', n):     # bare single digit = list marker
            continue
        g = month_group(n)
        if g is not None:
            if g in seen:
                continue
            seen.add(g)
        out.append(n)
    return out[:cap]

def build_prompt(card):
    ents   = json.loads(card['entities_for_prompt'])
    facts  = json.loads(card['fact_points'])
    target = int(card['target_words'])

    parts = [
        'اكتب تقريراً إخبارياً بالعربية الفصحى عن الموضوع التالي.',
        '',
        f'الموضوع: {card["topic_core"]}',
        '',
        'الكيانات التي يجب أن يذكرها التقرير:',
        '، '.join(ents),
        '',
        'الحقائق الأساسية التي يجب تغطيتها:',
    ]
    for f in facts:
        parts.append(f'- {f}')

    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            # slight emphasis: Qwen tends to drop numbers; ask once, clearly, without over-pushing
            parts += ['', 'احرص على ذكر هذه الأرقام والتواريخ كما هي: ' + '، '.join(nums)]
    if card['has_agencies']:
        ags = json.loads(card['source_agencies'])
        parts += ['', 'انسب المعلومات إلى: ' + '، '.join(ags)]
    if card['has_quotes']:
        parts += ['', 'أدرج تصريحات منسوبة للأطراف المعنية، بصياغتك أنت.']

    hints = random.sample(STYLE_HINTS, k=random.choice([3, 4]))
    parts += ['', 'إرشادات التحرير:'] + [f'- {h}' for h in hints]
    parts += ['',
              f'الطول: لا يقل التقرير عن {target} كلمة ولا يزيد عن {int(target*1.12)} كلمة. '
              f'اكتب تقريراً مكتملاً ضمن هذا النطاق.',
              '',
              'اكتب نص التقرير مباشرة: دون عنوان، ودون أي تنسيق (لا نجوم ** ولا رموز تنسيق)، '
              'ودون مقدمة أو تعليق منك.']
    return '\n'.join(parts)

## Formatting normalization (shared by both classes)

The model tends to wrap articles in a bold headline and split them into paragraphs; my human
corpus has none of that. I strip formatting artifacts and collapse newlines so the raw AI output
already matches the human single-paragraph shape. NB3 applies the identical function to both
classes, keeping the treatment symmetric.

In [4]:
def normalize_format(text):
    t = str(text)
    t = re.sub(r'<think>.*?</think>', '', t, flags=re.S)   # Qwen: strip any reasoning trace
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t)               # bold
    t = re.sub(r'__(.+?)__', r'\1', t)
    t = re.sub(r'(?<!\w)\*(.+?)\*(?!\w)', r'\1', t)      # italic
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.M)          # headings
    t = re.sub(r'^\s*[-–—>]\s+', '', t, flags=re.M)   # bullets / quotes
    t = re.sub(r'^\s*[-*_]{3,}\s*$', '', t, flags=re.M)   # rules
    t = re.sub(r'\n+', ' ', t)                            # human text has no newlines
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()

## Decoding parameters and generation

GPT accepts `temperature` (0-2) alongside `top_p` and the penalties — unlike Sonnet 5 / Opus 4.8,
which reject sampling knobs entirely. So per-card decoding variation is back for this generator,
same ranges my DeepSeek and Qwen notebooks used.

Two things I carry over from hard-won experience:

- **Reasoning off.** GPT-5 Mini is a reasoning model; I disable it via OpenRouter's `reasoning`
  flag. Reasoning tokens are billed as output and would eat into `max_tokens`, silently truncating
  long reports — exactly the trap I hit with Sonnet 5's adaptive thinking.
- **`max_tokens` at 4.5x target words.** Arabic runs ~3.5 tok/word and I'd rather overshoot the
  ceiling than truncate an article; it's a cap, not a charge.

In [5]:
def sample_params():
    return {
        'temperature':       round(random.uniform(0.8, 1.1), 3),
        'top_p':             0.95,
        'frequency_penalty': round(random.uniform(0.2, 0.5), 3),
        'presence_penalty':  0.1,
    }

def _call(messages, params, max_tokens, max_retries=5):
    # Empty completions are NOT network errors, so I retry them immediately (no backoff);
    # only real network/API errors get a short capped wait.
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME, messages=messages, max_tokens=max_tokens,
                timeout=90.0,
                extra_body={'reasoning': {'effort': 'low', 'exclude': True}},    # no reasoning tokens
                **params)
            content = resp.choices[0].message.content
            if not content or not content.strip():
                last_err = RuntimeError('empty completion')
                continue
            return content.strip(), resp.usage
        except Exception as e:
            last_err = e
            if attempt == max_retries - 1:
                raise
            time.sleep(min(2 ** attempt, 8))
    raise last_err

def generate_article(card, params):
    prompt = build_prompt(card)
    target = int(card['target_words'])
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt}]
    text, usage = _call(msgs, params, int(target * 4.5) + 800)
    text = normalize_format(text)
    tin, tout = usage.prompt_tokens, usage.completion_tokens

    n = len(text.split())
    if n < target * 0.88:
        need = int((target - n) * 0.9)          # measured, to avoid overshoot
        msgs2 = msgs + [{'role': 'assistant', 'content': text},
                        {'role': 'user',
                         'content': f'أضف نحو {need} كلمة فقط لإكمال التقرير بنفس الأسلوب، '
                                    f'دون تكرار ما ورد ودون تنسيق. اكتب التتمة فقط.'}]
        try:
            ext, usage2 = _call(msgs2, params, int(need * 4.5) + 400)
            text = normalize_format(text + ' ' + normalize_format(ext))
            tin += usage2.prompt_tokens; tout += usage2.completion_tokens
        except Exception:
            pass
    return text, tin, tout

## Acceptance gate (fuzzy, prefix-tolerant, mirror-rule aware)

Accept only if the article anchors to the same event: weighted coverage >= 80% AND entity coverage
>= 65% (hard floor). Only elements the source actually had are scored. Matching normalizes clitics,
`ال`, diacritics, alef/ya/ta-marbuta and Arabic-Indic vs Western digits; a month is satisfied by
either calendar name.

In [6]:
_AR_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩',
                           '0123456789')

def _norm(s):
    s = re.sub(r'[\u064B-\u0652]', '', s)                # diacritics
    s = s.translate(_AR_DIGITS)
    return (s.replace('أ','ا').replace('إ','ا').replace('آ','ا')
             .replace('ة','ه').replace('ى','ي'))

def _norm_entity(e):
    e = _norm(e)
    e = re.sub(r'^[وفبكل]?ال', '', e)
    e = re.sub(r'^لل', '', e)
    e = re.sub(r'^[وفبكل](?=.{3,})', '', e)
    return e.strip()

def _present(item, tn, is_entity=True):
    n = _norm_entity(item) if is_entity else _norm(item)
    return len(n) >= 2 and n in tn

def _number_present(item, tn):
    g = month_group(item)
    if g is not None:
        names = MONTH_GROUPS[g] | {k for k, v in AMBIG.items() if v == g}
        return any(_norm(m) in tn for m in names)
    return _present(item, tn, is_entity=False)

def acceptance_gate(article, card, W_ENT=3, W_NUM=2, W_AG=1):
    tn = _norm(article)
    scores, weights, detail = [], [], {}

    ents = json.loads(card['entities_for_prompt'])
    if ents:
        hit = sum(_present(e, tn) for e in ents)
        ent_cov = hit / len(ents)
        detail['entities'] = f'{hit}/{len(ents)}'
        scores.append(ent_cov); weights.append(W_ENT)
    else:
        ent_cov = 1.0
        detail['entities'] = 'none'

    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            hit = sum(_number_present(n, tn) for n in nums)
            c = hit / len(nums); detail['numbers'] = f'{hit}/{len(nums)}'
            scores.append(c); weights.append(W_NUM)
    if card['has_agencies']:
        ags = json.loads(card['source_agencies'])
        hit = sum(_present(a, tn, False) for a in ags)
        c = hit / max(len(ags), 1); detail['agencies'] = f'{hit}/{len(ags)}'
        scores.append(c); weights.append(W_AG)

    weighted = sum(s * w for s, w in zip(scores, weights)) / max(sum(weights), 1)
    passed = (weighted >= 0.80) and (ent_cov >= 0.65)
    return {'passed': passed, 'weighted': round(weighted, 3),
            'entity_cov': round(ent_cov, 3), 'detail': detail}

def length_ok(article, target, tol=0.15):
    n = len(article.split())
    return abs(n - target) / target <= tol, n

## Smoke test — one live call before the pilot

Cheap insurance I added after Sonnet/Opus errored on all 470 requests twice in a row over bad
decoding params. GPT-5 Mini is a reasoning model, so `temperature` and `max_tokens` could behave
differently than on DeepSeek/Qwen; one call tells me in seconds instead of after 30 pilot articles
of retry-backoff.

In [7]:
_c = my_cards.iloc[0]
try:
    _t, _ti, _to = generate_article(_c, sample_params())
    print('SMOKE TEST OK —', MODEL_NAME, 'accepted the params')
    print(f'  tokens: {_ti} in / {_to} out | words: {len(_t.split())} (target {int(_c["target_words"])})')
    print('  md left:', '**' in _t, '| newlines left:', chr(10) in _t)
    print('  sample:', _t[:150].replace(chr(10), ' '), '...')
    SMOKE_OK = True
except Exception as _e:
    SMOKE_OK = False
    print('SMOKE TEST FAILED — fix params before running the pilot. Real error:')
    print(' ', type(_e).__name__, ':', str(_e)[:400])

SMOKE TEST OK — openai/gpt-5-mini accepted the params
  tokens: 2690 in / 2200 out | words: 861 (target 898)
  md left: False | newlines left: False
  sample: قدّم مسلسل "هجمة مرتدة" صورة لسعد الدين إبراهيم بوصفه "جاسوساً"، على الرغم من أن القضاء المصري كان قد برّأه من كل ما نُسب إليه من تهم في حكم لمحكمة ال ...


## Phase A — pilot (30 cards, synchronous)

Generates 30, prints each live, saves after each, reports gate/length/cost and a formatting-artifact
check. Nothing here is destructive; I read the results before spending on the rest.

In [8]:
# Phase A (pilot): runs ONLY when PILOT_ONLY is True.
# In production/resume mode it is skipped entirely — no wasted pilot regeneration.
PILOT_CKPT = f'{OUT_DIR}/pilot_{GENERATOR_ID}.parquet'
p = None

def _run_pilot():

    pilot = my_cards.head(PILOT_N)
    rows, t0 = [], time.time()
    tok_in = tok_out = 0

    print(f'--- pilot: {PILOT_N} articles ---', flush=True)
    for i, card in pilot.iterrows():
        a0 = time.time()
        try:
            params = sample_params()
            text, tin, tout = generate_article(card, params)
            gate = acceptance_gate(text, card)
            lok, nwords = length_ok(text, int(card['target_words']))
            tok_in += tin; tok_out += tout
            rows.append({'pair_id': card['pair_id'], 'passed': gate['passed'],
                         'weighted': gate['weighted'], 'entity_cov': gate['entity_cov'],
                         'len_ok': lok, 'words': nwords, 'target': int(card['target_words']),
                         'detail': str(gate['detail']), 'text': text})
            pd.DataFrame(rows).to_parquet(PILOT_CKPT, index=False)          # save after every article
            flag = 'OK ' if (gate['passed'] and lok) else 'BAD'
            print(f'[{len(rows):2d}/{PILOT_N}] {flag} {card["pair_id"]} | '
                  f'cov={gate["weighted"]:.0%} ent={gate["entity_cov"]:.0%} '
                  f'| {nwords}w/{int(card["target_words"])}w ({nwords/int(card["target_words"]):.2f}) '
                  f'| md={chr(34)+chr(34) in text} | {time.time()-a0:.0f}s | {gate["detail"]}',
                  flush=True)
        except Exception as e:
            print(f'[{len(rows):2d}/{PILOT_N}] ERR {card["pair_id"]}: {type(e).__name__}: {str(e)[:120]}',
                  flush=True)

    p = pd.DataFrame(rows)
    print(f'\n=== pilot summary ===')
    print(f'generated  : {len(p)}/{PILOT_N} in {time.time()-t0:.0f}s')
    print(f'gate pass  : {p["passed"].sum()}/{len(p)} ({100*p["passed"].mean():.0f}%)')
    print(f'length ok  : {p["len_ok"].sum()}/{len(p)} ({100*p["len_ok"].mean():.0f}%)')
    print(f'entity cov : mean {p["entity_cov"].mean():.0%} | min {p["entity_cov"].min():.0%}')
    print(f'weighted   : mean {p["weighted"].mean():.0%}')
    print(f'words/target: mean {(p["words"]/p["target"]).mean():.2f}')
    cost = tok_in/1e6*PRICE_IN + tok_out/1e6*PRICE_OUT
    print(f'cost       : ${cost:.3f} | projected {N_TARGET}: ${cost/max(len(p),1)*N_TARGET:.2f}')

    n_md = int(p['text'].str.contains('**', regex=False).sum())
    n_nl = int(p['text'].str.contains(chr(10), regex=False).sum())
    print(f'\nmarkdown left: {n_md} | newlines left: {n_nl}   (both must be 0)')
    if n_md or n_nl:
        print('!! formatting artifacts present — do NOT run production until fixed')
    return p

if PILOT_ONLY:
    p = _run_pilot()
else:
    print('production/resume mode — pilot skipped, going straight to Phase B')

production/resume mode — pilot skipped, going straight to Phase B


In [9]:
# read one full article + list any failures (pilot mode only)
if p is not None and len(p):
    best = p.loc[p['weighted'].idxmax()]
    print('='*70); print('BEST', best['pair_id'], best['detail'], f"{best['words']}w/{best['target']}w")
    print('='*70); print(best['text'][:1200])
    fails = p[~p['passed']]
    print(f'\nfailed: {len(fails)}')
    for _, f in fails.iterrows():
        print(f"  {f['pair_id']}: w={f['weighted']:.0%} ent={f['entity_cov']:.0%} {f['detail']}")

## Phase B — production (draw until QUOTA successes)

Saves after every single article, so a Kaggle timeout costs me nothing but time: set `RESUME_PATH`
to the partial parquet and the run picks up its banked successes and carries on. The loop stops the
moment `n_success` hits QUOTA — rejects never count toward it.

In [10]:
def make_record(card, text, params, gate, nwords, lok, attempt, error=''):
    return {
        'id': f'AI_{GENERATOR_ID}_{card["pair_id"]}',
        'text': text, 'label': 'ai', 'generator': GENERATOR_ID,
        'source_pair_id': card['pair_id'],
        'temperature': params.get('temperature', 0), 'top_p': params.get('top_p', 0),
        'frequency_penalty': params.get('frequency_penalty', 0),
        'presence_penalty': params.get('presence_penalty', 0),
        'target_words': int(card['target_words']), 'actual_words': nwords,
        'entities_injected': card['entities_for_prompt'],
        'coverage_weighted': gate.get('weighted', 0), 'coverage_entities': gate.get('entity_cov', 0),
        'gate_passed': bool(gate.get('passed', False)), 'length_ok': bool(lok),
        'generation_attempts': attempt, 'error': error,
    }

In [11]:
CKPT         = f'{OUT_DIR}/aig_{GENERATOR_ID}.parquet'
MAX_ATTEMPTS = 2
# To RESUME after a timeout: upload the partial aig_qwen.parquet as a dataset and set the path.
RESUME_PATH  = '/kaggle/input/aigt-aig-qwen-partial/aig_qwen.parquet'  # set None for a fresh run

if not PILOT_ONLY:
    done_ids, records = set(), []
    for pth in [RESUME_PATH, CKPT]:
        if pth and os.path.exists(pth):
            prev = pd.read_parquet(pth)
            records = prev.to_dict('records'); done_ids = set(prev['source_pair_id'])
            print(f'resuming: {len(done_ids)} already done', flush=True)
            break

    # resume: count successes already banked, and skip every card already processed
    done_pairs = {r['source_pair_id'] for r in records}
    n_success  = sum(1 for r in records if r.get('gate_passed'))
    todo = my_cards[~my_cards['pair_id'].isin(done_pairs)].reset_index(drop=True)
    print(f'quota {QUOTA} successes | already banked {n_success} | work list {len(todo)} remaining',
          flush=True)

    t0 = time.time(); n_rej = n_err = 0; tin_tot = tout_tot = 0
    for i, card in todo.iterrows():
        if n_success >= QUOTA:                    # STOP when the quota of SUCCESSES is met
            print(f'quota reached: {n_success} successes — stopping draw', flush=True)
            break
        try:
            best_rec, params = None, {}
            for attempt in range(1, MAX_ATTEMPTS + 1):
                params = sample_params()
                text, tin, tout = generate_article(card, params)
                tin_tot += tin; tout_tot += tout
                gate = acceptance_gate(text, card)
                lok, nwords = length_ok(text, int(card['target_words']))
                rec = make_record(card, text, params, gate, nwords, lok, attempt)
                if best_rec is None or (rec['gate_passed'], rec['coverage_weighted']) > \
                                       (best_rec['gate_passed'], best_rec['coverage_weighted']):
                    best_rec = rec                 # keep the best attempt
                if gate['passed'] and lok:
                    break
            rec = best_rec
            if rec['gate_passed']:
                n_success += 1                     # only PASSING articles count toward the quota
            else:
                n_rej += 1
        except Exception as e:
            rec = make_record(card, '', {}, {}, 0, False, 0, f'{type(e).__name__}: {str(e)[:150]}')
            n_err += 1
            print(f'ERR {card["pair_id"]}: {rec["error"]}', flush=True)

        records.append(rec)
        pd.DataFrame(records).to_parquet(CKPT, index=False)          # SAVE AFTER EVERY ARTICLE

        done = len(records)
        status = 'ERR' if rec['error'] else ('OK ' if rec['gate_passed'] else 'BAD')
        rate = n_success / max(time.time() - t0, 1e-6)
        eta  = (QUOTA - n_success) / max(rate, 1e-6) / 60
        print(f'[proc {done:4d} | ok {n_success:3d}/{QUOTA}] {status} {card["pair_id"]} | '
              f'cov={rec["coverage_weighted"]:.0%} ent={rec["coverage_entities"]:.0%} '
              f'| {rec["actual_words"]}w/{rec["target_words"]}w | att={rec["generation_attempts"]} '
              f'| bad={n_rej} err={n_err} | ETA {eta:.0f}m', flush=True)

        if done % 10 == 0:
            cost = tin_tot/1e6*PRICE_IN + tout_tot/1e6*PRICE_OUT
            print(f'    >>> processed {done} | {n_success}/{QUOTA} successes | cost so far ${cost:.2f}',
                  flush=True)

    cost = tin_tot/1e6*PRICE_IN + tout_tot/1e6*PRICE_OUT
    if n_success < QUOTA:
        print(f'\n!! WORK LIST EXHAUSTED before quota: {n_success}/{QUOTA}. Need more cards.', flush=True)
    print(f'\nDONE {GENERATOR_ID} | successes={n_success}/{QUOTA} | rejects={n_rej} err={n_err} '
          f'| processed={len(records)} | total cost ${cost:.2f}', flush=True)
else:
    print('PILOT_ONLY is True — production skipped. Review the pilot, then set PILOT_ONLY = False.')

quota 440 successes | already banked 0 | work list 884 remaining
[proc    1 | ok   1/440] OK  HA_02878 | cov=98% ent=96% | 799w/898w | att=1 | bad=0 err=0 | ETA 178m
[proc    2 | ok   2/440] OK  HA_00246 | cov=86% ent=77% | 455w/466w | att=2 | bad=0 err=0 | ETA 230m
[proc    3 | ok   3/440] OK  HA_02794 | cov=82% ent=97% | 714w/793w | att=1 | bad=0 err=0 | ETA 199m
[proc    4 | ok   4/440] OK  HA_01916 | cov=89% ent=77% | 695w/656w | att=2 | bad=0 err=0 | ETA 239m
[proc    5 | ok   5/440] OK  HA_02218 | cov=97% ent=94% | 618w/667w | att=2 | bad=0 err=0 | ETA 251m
[proc    6 | ok   6/440] OK  HA_02857 | cov=86% ent=83% | 869w/981w | att=1 | bad=0 err=0 | ETA 232m
[proc    7 | ok   7/440] OK  HA_01788 | cov=98% ent=96% | 624w/642w | att=1 | bad=0 err=0 | ETA 214m
[proc    8 | ok   8/440] OK  HA_00195 | cov=92% ent=87% | 406w/441w | att=1 | bad=0 err=0 | ETA 199m
[proc    9 | ok   9/440] OK  HA_01006 | cov=86% ent=71% | 490w/529w | att=1 | bad=0 err=0 | ETA 190m
[proc   10 | ok   9/440] B

In [12]:
# final verification (production only)
if not PILOT_ONLY:
    final = pd.read_parquet(CKPT)
    n_pass = int(final['gate_passed'].sum())
    n_fail = int((~final['gate_passed']).sum())
    print('processed    :', len(final), '| unique ids:', final['source_pair_id'].nunique())
    print('SUCCESSES    :', n_pass, f'(quota was {QUOTA})')
    print('rejects      :', n_fail, '(these carry over to the next generator)')
    print('errors       :', int((final['error'] != '').sum()))
    print('length ok    :', int(final['length_ok'].sum()), f"({100*final['length_ok'].mean():.0f}%)")
    print('mean coverage:', f"{final['coverage_weighted'].mean():.0%}")
    print('mean attempts:', f"{final['generation_attempts'].mean():.2f}")
    n_md = int(final['text'].str.contains('**', regex=False).sum())
    n_nl = int(final['text'].str.contains(chr(10), regex=False).sum())
    print('markdown/newlines left:', n_md, '/', n_nl, '(must be 0)')
    print(f'\nquota met: {n_pass >= QUOTA}  |  upload as aigt-aig-qwen for the next generator')

processed    : 461 | unique ids: 461
SUCCESSES    : 440 (quota was 440)
rejects      : 21 (these carry over to the next generator)
errors       : 0
length ok    : 460 (100%)
mean coverage: 93%
mean attempts: 1.13
markdown/newlines left: 0 / 0 (must be 0)

quota met: True  |  upload as aigt-aig-qwen for the next generator


## Notes

- **Synchronous, not Batch, and that's a feature.** OpenRouter has no batch endpoint, which brings
  back the regenerate loop. Measured per-card pass rates: DeepSeek 97.6% and Qwen 97.6% with the
  loop, versus Sonnet 92.8% and Opus 80.9% through Batch without it. The loop is worth more than
  Batch's 50% discount at these prices.
- **Efficiency tier on purpose.** Opus (stronger) failed 90/470 where Sonnet (weaker) failed 34/470,
  because the stronger model paraphrases figures instead of carrying them. GPT-5 Mini is chosen for
  literal instruction-following, not benchmark scores.
- **Number emphasis stays.** "احرص على ذكر ... كما هي" lifted Qwen's gate 76% -> 93%, and salvaged
  75% of the Sonnet/Opus rejects that had been at 0%. It's factual fidelity — making the article
  carry its card's facts — not tuning against any of my five detection features.
- **Kaggle's 12h limit** is the real constraint here, not money. ~440 successes at ~20-40s each is
  roughly 3-5 hours, so one Save Version run should finish; if it doesn't, RESUME_PATH exists.
- **Rejects flow to Gemini**, which runs next and takes everything still unsalvaged plus every card
  I never reach.